<a href="https://colab.research.google.com/github/justorfc/Estadistica_Aplicada_con_Python_y_R_2026_2/blob/main/13_Semana_13_Introducci%C3%B3n_a_Series_de_Tiempo_y_Descomposici%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este Notebook es la propuesta estructurada para la **Semana 13**, marcando el inicio del **Eje V: Series de Tiempo y Proyecto Integrador**. En esta semana, cambiamos el paradigma: el orden de los datos ahora importa, ya que introducimos el tiempo como variable fundamental para analizar ciclos agroclimáticos (estacionalidad) y cambios a largo plazo (tendencia).

# Semana 13: Introducción a Series de Tiempo y Descomposición

**Resultado de aprendizaje:** Analiza componentes temporales en series agroclimáticas e hídricas, separando matemáticamente la tendencia, la estacionalidad y el ruido aleatorio para comprender la dinámica del sistema.

---

#### Sesión 1: El Índice Temporal y la Descomposición Aditiva (80 - 90 minutos)

**Objetivo:** Comprender la estructura de una serie de tiempo, configurar correctamente los índices de fechas en Python y descomponer la serie en sus componentes fundamentales.

* **20 min - Diálogo socrático y conceptualización gráfica (Lápiz y papel):**
* *Situación:* Si medimos la Evapotranspiración (ETo) todos los meses durante 10 años, ¿esperamos que sea constante? No, subirá en verano y bajará en invierno (Estacionalidad). Y si hay cambio climático, el promedio general irá subiendo año tras año (Tendencia).
* *Actividad:* Los estudiantes dibujan un eje X (Tiempo en años) y un eje Y (ETo). Trazan a mano alzada una onda que sube y baja rítmicamente, pero que en general va inclinada hacia arriba. Se discute cómo la máquina separa esa onda compleja en tres piezas simples.


* **45 min - Exploración en Google Colab (Python):**
* Carga del cuaderno de la semana 13.
* Creación y manejo de índices temporales con `pandas` (`pd.to_datetime`).
* Uso de `statsmodels.tsa.seasonal_decompose` para separar la serie en: Observado, Tendencia, Estacionalidad y Residuos.


* **15 min - Reflexión manuscrita:**
* Interpretación técnica: ¿Qué nos dice la gráfica de "Estacionalidad" aislada sobre los meses de mayor demanda hídrica para programar el riego?



---

#### Sesión 2: Objetos Temporales y Transición a R (80 - 90 minutos)

**Objetivo:** Trasladar la manipulación de fechas y la descomposición temporal al entorno de R, comprendiendo la diferencia entre un vector normal y un objeto de serie de tiempo (`ts` o `tsibble`).

* **20 min - La naturaleza del tiempo en R:**
* Explicación de cómo R maneja las frecuencias temporales (ej. frecuencia = 12 para datos mensuales). Introducción al objeto `ts`.


* **25 min - Prompts para descomposición clásica en R:**
* Demostración de cómo instruir al asistente de IA para convertir un dataframe a un objeto `ts` y utilizar la función nativa `decompose()` o la más robusta `stl()` (Seasonal Decomposition of Time Series by Loess).


* **40 min - Reto en Posit Cloud:**
* Los estudiantes ejecutan el flujo en su documento RMarkdown. Generan el gráfico de los 4 paneles de descomposición en R, extraen el mes de mayor consumo de agua según el componente estacional, y documentan su razonamiento en la bitácora de IA.



---

A continuación, el contenido listo para integrarse en las celdas de tu cuaderno de Google Colab.

---

### Celda de Texto 1

# Semana 13: Series de Tiempo y Descomposición
**Asignatura:** Estadística Aplicada con Python y R  
**Programa:** Ingeniería Agrícola - Universidad de Sucre  
**Profesor:** Justo Rafael Fuentes Cuello  

---

### Situación de Interés: La Demanda Hídrica en el Tiempo
Hasta ahora, hemos asumido que el orden en que recolectamos los datos no importa (podíamos mezclar las filas de nuestro dataframe y el promedio o la regresión seguían siendo los mismos).

En las **Series de Tiempo**, el orden lo es todo. Hoy analizaremos un registro simulado de 10 años de Evapotranspiración (ETo) mensual. En hidrología y agricultura, predecir el futuro requiere entender cómo se comporta la variable rítmicamente cada año (**Estacionalidad**) y si el consumo general está aumentando o disminuyendo con el paso de los años (**Tendencia**).

```

### Celda de Código 1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose

sns.set_theme(style="whitegrid")
np.random.seed(42)

# Simulamos 10 años de datos mensuales (120 meses) desde Enero de 2016
fechas = pd.date_range(start='2016-01-01', periods=120, freq='ME')

# Construimos los componentes matemáticos de la serie
# 1. Tendencia: Aumento leve por año (ej. por aumento gradual de temperatura)
tendencia = np.linspace(100, 130, 120)

# 2. Estacionalidad: Ciclo anual (senoidal) que sube en verano y baja en invierno
meses_del_año = np.arange(120) % 12
estacionalidad = 30 * np.sin(2 * np.pi * meses_del_año / 12)

# 3. Ruido: Variabilidad climática aleatoria (El Niño, La Niña, frentes fríos anómalos)
ruido = np.random.normal(loc=0, scale=8, size=120)

# Serie observada = Tendencia + Estacionalidad + Ruido (Modelo Aditivo)
eto_observada = tendencia + estacionalidad + ruido

# Creamos el DataFrame
df_serie = pd.DataFrame({
    'Fecha': fechas,
    'ETo_mm': eto_observada
})

print("Serie de tiempo de Evapotranspiración generada (2016 - 2025).")
df_serie.head()

### 1. El Índice Temporal (Datetime Index)
Para que Python (pandas) entienda que está tratando con una serie de tiempo y no con simples números ordenados, debemos convertir la columna de fechas en el **Índice (Index)** del DataFrame. Esto habilita operaciones avanzadas de series temporales.

```

### Celda de Código 2

In [ ]:
# Convertimos la columna Fecha en el índice del dataframe
df_serie.set_index('Fecha', inplace=True)

# Graficamos la serie original (Observada)
plt.figure(figsize=(12, 4))
plt.plot(df_serie.index, df_serie['ETo_mm'], color='dodgerblue', linewidth=2)
plt.title('Evapotranspiración Mensual Observada (2016 - 2025)')
plt.xlabel('Año')
plt.ylabel('ETo (mm/mes)')
plt.show()

### 2. Descomposición Matemática de la Serie
La gráfica anterior es la realidad cruda. Ahora aplicaremos un algoritmo de descomposición estacional (`seasonal_decompose`) de `statsmodels` para obligar a la máquina a separar la serie en sus tres "piezas" constituyentes:
1.  **Trend (Tendencia):** La dirección a largo plazo.
2.  **Seasonal (Estacionalidad):** El patrón repetitivo puro de cada año.
3.  **Residuo (Ruido):** Lo que queda después de quitar la tendencia y la estacionalidad (anomalías).

```

### Celda de Código 3

In [ ]:
# Realizamos la descomposición asumiendo un modelo aditivo
# period=12 indica que el ciclo se repite cada 12 meses
descomposicion = seasonal_decompose(df_serie['ETo_mm'], model='additive', period=12)

# Extraemos las partes
tendencia_extraida = descomposicion.trend
estacional_extraida = descomposicion.seasonal
residuo_extraido = descomposicion.resid

# Graficamos los 4 paneles de forma estructurada
fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)

descomposicion.observed.plot(ax=axes[0], color='black')
axes[0].set_ylabel('Observado')
axes[0].set_title('Descomposición Estacional de la Evapotranspiración')

descomposicion.trend.plot(ax=axes[1], color='red')
axes[1].set_ylabel('Tendencia')

descomposicion.seasonal.plot(ax=axes[2], color='green')
axes[2].set_ylabel('Estacionalidad')

descomposicion.resid.plot(ax=axes[3], color='purple', style='o', markersize=3)
axes[3].axhline(0, color='black', linestyle='--')
axes[3].set_ylabel('Residuos')

plt.xlabel('Fecha')
plt.tight_layout()
plt.show()

### 3. Aislamiento e Interpretación del Ciclo Estacional
Para planificar el calendario de riego, un ingeniero necesita aislar la firma estacional pura. Vamos a tomar un solo año del componente estacional para ver matemáticamente cuántos milímetros de agua extra (o de menos) aporta la estacionalidad climática en cada mes.

```

### Celda de Código 4

In [ ]:
# Extraemos el patrón estacional de un solo año (ej. 2017)
patron_anual = estacional_extraida.loc['2017-01-01':'2017-12-31']
meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

plt.figure(figsize=(8, 4))
sns.barplot(x=meses, y=patron_anual.values, color='lightgreen')
plt.title('Firma Estacional: Aporte neto mensual a la ETo')
plt.xlabel('Mes')
plt.ylabel('Desviación de ETo (mm) por Estacionalidad')
plt.axhline(0, color='black')
plt.show()

### 🛑 Reflexión y Reserva Cognitiva (Síntesis manuscrita)
Toma tu cuaderno de apuntes y analiza las gráficas:
1. Revisa el panel de **Tendencia (rojo)** en el gráfico de 4 paneles. A pesar de que los picos en el panel "Observado" suben y bajan violentamente, la línea roja asciende suavemente. ¿Qué conclusión técnica sobre la demanda hídrica a largo plazo sacarías de esta línea recta?
2. Observa la **Firma Estacional** (el gráfico de barras verdes). Identifica los meses con valores negativos. ¿Significa esto que la ETo real en esos meses es un número negativo? (¡Recuerda que la serie original = tendencia + estacionalidad + residuo!). Explica qué significa realmente un aporte estacional negativo.
3. Si un residuo en un mes específico (ej. mayo 2020) es de $+20 \text{ mm}$, esto es ruido aleatorio. Agronómicamente, ¿qué fenómeno físico extremo pudo haber causado que la ETo fuera 20 mm más alta de lo que la tendencia y la estacionalidad predecían?

---

### Instrucciones para el reto en R (Trabajo Autónomo y Sesión 2)

**Misión:** R es considerado por muchos estadísticos como el rey indiscutible de las series de tiempo. Tu reto es aplicar el concepto de indexación temporal y descomposición (STL o decompose clásica) en **Posit Cloud**.

**Pasos a seguir:**
1. Abre tu proyecto en Posit Cloud y crea un nuevo documento RMarkdown (o Quarto).
2. Utiliza este *prompt* con tu asistente de IA (ChatGPT, Gemini, etc.):
   > *"Actúa como un profesor experto en series de tiempo en R. En mi clase con Python simulamos una serie mensual de 10 años de Evapotranspiración y aplicamos `seasonal_decompose` de `statsmodels` para ver la serie observada, la tendencia, la estacionalidad y el residuo. Necesito replicar esto en R. Genera código para simular la serie y enséñame cómo convertirla en un objeto de serie temporal usando la función `ts(datos, start = c(2016, 1), frequency = 12)`. Luego, muéstrame cómo aplicarle la función `decompose()` y usar `plot()` sobre el resultado para obtener el gráfico de 4 paneles. Explica el código paso a paso para documentarlo en RMarkdown."*
3. Experimenta en R: en lugar de la función base `decompose()`, pregúntale al chatbot cómo se haría la misma gráfica usando la función más avanzada `stl(serie, s.window = 'periodic')`. Notarás que `stl` utiliza regresión LOESS internamente, lo cual vimos en semanas anteriores.
4. **Entrega:** Renderiza el RMarkdown a HTML/PDF. En tu "Bitácora de IA", reporta si te pareció más fácil el manejo de fechas en Python (pandas `pd.to_datetime`) o la creación del objeto `ts()` directamente en R.